## Objectives

- Understand the distribution of numerical and categorical variables in the cleaned dataset.
- Identify missing values, skewness, and potential outliers that may affect modelling.
- Explore the distribution of the target variable (SalePrice) and determine whether transformations (e.g. log) are needed.
- Investigate relationships between predictors and SalePrice using correlation and visual analysis.
- Generate insights that guide feature engineering and model selection.

## Inputs
* Cleaned dataset: 
    - `outputs/datasets/cleaned/CleanedData.csv`

## Outputs
- Transformed TrainSet and TestSet
- List of applied transformers:
    - Categorical encoding (ordinal + one-hot)
    - Log/Power/Box-Cox transformations
    - Winsorization (IQR)
    - Smart correlation selection


### Import Cell

In [ ]:
import os
import time
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats


from feature_engine.encoding import OneHotEncoder
from feature_engine.selection import SmartCorrelatedSelection
from feature_engine.outliers import Winsorizer
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer

sns.set(style="whitegrid")

---

## Change working directory

We need to set the current working directory to the parent folder for consistency.

In [ ]:
BASE_DIR = Path(r"C:\Users\david\Portfolio 5\heritage-housing")
os.chdir(BASE_DIR)
print("Working directory set to:", os.getcwd())

## Load Cleaned Data

We load the cleaned dataset and split it into Train and Test sets.



This notebook loads the cleaned dataset and splits it into train and test sets. This keeps the workflow consistent with the rest of the project and avoids data leakage.

In [ ]:
# Load cleaned data
data_path = BASE_DIR / "outputs/datasets/cleaned/CleanedData.csv"
df = pd.read_csv(data_path)

print("Loaded dataset shape:", df.shape)



In [ ]:
from src.data.split import split_data

X_train, X_test, y_train, y_test = split_data(df)

---

# ===============================
## Phase 1: Exploration (Analysis only)
# ===============================

## Helper Functions

These functions help check missing values and generate diagnostic plots for numeric and categorical features.

In [ ]:
def check_missing_values(df):
    """
    Print a summary of missing values per column in the dataframe.
    """
    missing = df.isna().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print("* No missing values found.")
    else:
        print("* Missing values found:")
        print(missing)


def diagnostic_plots(df, col):
    """
    Generate histogram, QQ plot, and boxplot for a numeric column.
    """
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    sns.histplot(df[col], kde=True, ax=axes[0])
    stats.probplot(df[col], dist="norm", plot=axes[1])
    sns.boxplot(x=df[col], ax=axes[2])
    axes[0].set_title("Histogram")
    axes[1].set_title("QQ Plot")
    axes[2].set_title("Boxplot")
    fig.suptitle(col, fontsize=18, y=1.05)
    plt.tight_layout()
    plt.show()


def diagnostic_plots_cat(df, col):
    """
    Generate a count plot for a categorical column.
    """
    plt.figure(figsize=(4,3))
    sns.countplot(data=df, x=col, order=df[col].value_counts().index)
    plt.title(col)
    plt.xticks(rotation=90)
    plt.show()

## Identity Variables for Feature Engineering

### Phase 1: Transformation Exploration & Selection

In [ ]:
numeric_skewed = ['GrLivArea', 'LotArea', 'TotalBsmtSF', '1stFlrSF', 'BsmtFinSF1']

df_engineering = X_train[numeric_skewed].copy()

check_missing_values(df_engineering)

### Numerical Transformation Evaluation (Skewness Comparison)

In [ ]:
analysis = []

for col in numeric_skewed:
    row = {"Feature": col}

    row["Original Skew"] = X_train[col].skew()
    row["Log1p Skew"] = np.log1p(X_train[col]).skew()

    analysis.append(row)

skew_df = pd.DataFrame(analysis)
skew_df

In [ ]:
# Prepare data
x = np.arange(len(skew_df["Feature"]))
width = 0.35

plt.figure(figsize=(10,5))

plt.bar(x - width/2, skew_df["Original Skew"], width, label="Original Skew")
plt.bar(x + width/2, skew_df["Log1p Skew"], width, label="Log1p Skew")

plt.xticks(x, skew_df["Feature"], rotation=45)
plt.ylabel("Skewness")
plt.title("Feature Skewness Before vs After Transformation")
plt.legend()
plt.tight_layout()
plt.show()

The plot shows that applying log-based transformations significantly reduced skewness across all selected numerical features. This improves distribution symmetry, reduces the influence of extreme values, and supports more stable model training.
This reduces bias in linear models and stabilises variance, improving regression performance

### Ordinal Categorical Variables

- `KitchenQual` has a natural order, so we will encode it numerically. is an ordinal feature with a clear ranking (Poor to Excellent). Ordinal encoding is used to preserve this order, enabling the model to capture the relationship between kitchen quality and house price. One-hot encoding was avoided because it removes ordinal structure by treating categories as independent.

### Numeric Variables for Transformation:
- `GrLivArea`, `LotArea`, `TotalBsmtSF`, `1stFlrSF`, `BsmtFinSF1`  
These features are right-skewed or have outliers and could potentially benefit from log/power/Box-Cox transformations.

### Variables for Correlation-Based Reduction:
- All numeric features  
To reduce multicollinearity and  thus keep only the most informative features.

---

In [ ]:
ordinal_vars = ['KitchenQual']


### Feature Engineering Decisions

Based on exploratory analysis, the following transformations were selected for the final pipeline:

- Ordinal encoding for ordered categorical variables (KitchenQual) to preserve ranking information
- One-hot encoding for nominal categorical variables
- Power transformations (Box-Cox / Yeo-Johnson) to reduce skewness in numerical features
- Winsorization (IQR-based) to reduce the impact of extreme outliers while retaining all observations
- Smart Correlation Selection (ρ > 0.8) to reduce multicollinearity and improve model stability

These transformations were chosen because they directly address key data issues identified during EDA, resulting in a dataset that is more stable, interpretable, and suitable for regression modelling.

# ===============================
## Phase 2: Final Pipeline (Model Ready)
# ===============================

### Feature Engineering Pipeline
All encoding, transformations, outlier handling, and feature selection are applied via a reusable pipeline function in `src`.


In [ ]:
from src.data.feature_engineering import feature_engineering_pipeline

X_train, X_test, fe_metadata = feature_engineering_pipeline(X_train, X_test)

print("* Feature engineering pipeline applied")
print(fe_metadata)
print(X_train.shape, X_test.shape)

### Smart Correlation 

- Spearman correlation is used as the method because the dataset contains skewed numerical variables with outliers, and we are interested in monotonic relationships rather than strictly linear dependencies.
- Correlation analysis is used to identify redundant features and reduce multicollinearity before modeling.
- Features with Spearman correlation > 0.8 are considered highly collinear, and one of each pair is removed based on domain relevance. This reduces multicollinearity, which can inflate variance in linear models, reduce interpretability, and increase overfitting risk.

In [ ]:
## Smart Correlation – Before vs After
plt.figure(figsize=(12,8))
sns.heatmap(X_train.corr(numeric_only=True),
            cmap="coolwarm",
            center=0)
plt.title("Correlation Matrix before Smart Correlation Selection")
plt.show()

In [ ]:
print("* Smart correlation selection applied!")
print(fe_metadata["dropped_features"])

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(X_train.corr(numeric_only=True),
            cmap="coolwarm",
            center=0)
plt.title("Correlation Matrix after Smart Correlation Selection")
plt.show()

In [ ]:
print(X_train.shape, X_test.shape)
print(list(X_train.columns) == list(X_test.columns))

## Apply Feature Engineering Transformers

### Box-Cox
Box-Cox selected for strongly right-skewed strictly positive variables. This helps to improve the regression model performance and stability.

### Yeo-Johnson

Yeo-Johnson selected for skewed variables (allows zeros). It improves distribution normality and helps enhance model performance.

### Outlier Handling (Winsorization)

Winsorization was applied to limit the influence of extreme values in key numeric features while retaining all observations in the dataset.

Housing data often contains unusually large or expensive properties that are still valid but can disproportionately influence models such as regression-based or tree-based algorithms. These extreme values can distort relationships between features and sale price, leading to less stable and less generalisable predictions.

Winsorization was preferred over removing outliers because:
- The dataset is relatively small, so removing data would risk losing valuable information.
- Extreme values may represent legitimate luxury properties rather than data errors.
- The business problem requires predicting prices across a wide range of homes, including high-value properties.

By capping extreme values using an IQR-based approach, the model becomes more robust to skewed distributions while still preserving the overall structure of the dataset. Tree-based and linear models are sensitive to extreme values, which can distort regression coefficients and split decisions.

Apply IQR-based Winsorization to selected numeric features.

---

In [ ]:
# Evaluate Winsorization visually
for col in fe_metadata["winsor_vars"]:
    diagnostic_plots(X_train, col)
    diagnostic_plots(X_test, col)

### Numerical Transformations 
- Apply Power Transformations to reduce skewness
- Box-Cox applied to strictly positive features
- Yeo-Johnson applied to all remaining skewed features 

### Transformation Summary
Final selected transformations are implemented in Phase 2 pipeline.

In [ ]:
print("=== FINAL FEATURE ENGINEERING SUMMARY ===")

print("\nOrdinal encoded variables:")
print(ordinal_vars)

print("Box-Cox:", fe_metadata["boxcox_vars"])
print("Yeo-Johnson:", fe_metadata["yeojohnson_vars"])
print("Winsorized:", fe_metadata["winsor_vars"])

print("\nCorrelation-based dropped features:")
print(fe_metadata["dropped_features"])

These transformations were selected over simpler preprocessing approaches because they directly address skewness, categorical ordering, and multicollinearity. This results in a more stable and interpretable dataset, improving model robustness and generalisation performance compared to using raw or minimally processed features.

# ===============================
## Phase 3: Final Outputs
# ===============================

In [ ]:
# Rebuild Train/Test sets for export
TrainSet_final = X_train.copy()
TrainSet_final["SalePrice"] = y_train.values

TestSet_final = X_test.copy()
TestSet_final["SalePrice"] = y_test.values

print(TrainSet_final.shape, TestSet_final.shape)
print(y_train.shape, y_test.shape)

In [ ]:
os.makedirs("outputs/datasets/processed", exist_ok=True)
TrainSet_final.to_csv(BASE_DIR / "outputs/datasets/processed/train_processed.csv", index=False)
TestSet_final.to_csv(BASE_DIR / "outputs/datasets/processed/test_processed.csv", index=False)